# Machine Learning Analysis

The goal of this section is to predict hourly food delivery demand using machine learning models.

Based on previous analysis, time-related features such as order hour and weekend indicators are expected to play a significant role, while weather conditions are included to evaluate their additional impact.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## Data Loading

In [7]:
orders = pd.read_csv("order_history_kaggle_data.csv")
weather = pd.read_excel("weather_data.xlsx")

## Data Preparation

In this step, datetime features are extracted and aligned between datasets. This allows us to merge order data with corresponding hourly weather observations and build a consistent dataset for modeling.

In [8]:
orders['order_datetime'] = pd.to_datetime(
    orders['Order Placed At'],
    format='%I:%M %p, %B %d %Y',
    errors='coerce'
)

weather['datetime'] = pd.to_datetime(weather['datetime'])

orders['hour'] = orders['order_datetime'].dt.floor('h')
weather['hour'] = weather['datetime'].dt.floor('h')

## Filtering Delivered Orders

Only delivered orders are used in this analysis, as they best represent actual completed demand. This ensures that the model focuses on real customer behavior.

In [9]:
orders = orders[orders['Order Status'] == 'Delivered']

## Merge Data

The order dataset is merged with weather data using hourly timestamps. This ensures that each order is associated with the weather conditions at the time it was placed.

In [10]:
df = pd.merge(
    orders,
    weather[['hour', 'temperature_2m', 'precipitation']],
    on='hour',
    how='inner'
)

## Aggregate Demand (Hourly)

The data is aggregated to hourly level, transforming individual orders into demand counts per hour.

In [11]:
df_ml = df.groupby('hour').agg(
    orders=('Order ID', 'count'),
    temperature=('temperature_2m', 'mean'),
    rain=('precipitation', 'mean')
).reset_index()

df_ml['order_hour'] = df_ml['hour'].dt.hour
df_ml['day_of_week'] = df_ml['hour'].dt.dayofweek
df_ml['is_weekend'] = (df_ml['day_of_week'] >= 5).astype(int)

## Features and Target

The target is hourly order count.

Features include time-based variables and weather variables to capture both behavioral and environmental effects.

In [12]:
X = df_ml[['order_hour', 'day_of_week', 'is_weekend', 'temperature', 'rain']]
y = df_ml['orders']

## Train-Test Split

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## Model 1: Linear Regression

Used as a baseline to understand simple relationships.

In [14]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

mae_lr = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr = r2_score(y_test, y_pred_lr)

print("Linear Regression")
print(f"MAE: {mae_lr:.2f}")
print(f"RMSE: {rmse_lr:.2f}")
print(f"R2: {r2_lr:.3f}")

Linear Regression
MAE: 4.57
RMSE: 6.68
R2: 0.219


### Interpretation

Linear Regression captures general trends but struggles with non-linear patterns such as peak hours and behavioral demand shifts.

This suggests that more flexible models may be needed.

## Model 2: Random Forest

In [15]:
rf = RandomForestRegressor(random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test, y_pred_rf)

print("Random Forest")
print(f"MAE: {mae_rf:.2f}")
print(f"RMSE: {rmse_rf:.2f}")
print(f"R2: {r2_rf:.3f}")

Random Forest
MAE: 3.90
RMSE: 5.67
R2: 0.438


Random Forest achieves lower RMSE and higher R² compared to Linear Regression.

This indicates that food delivery demand is influenced by non-linear relationships, such as sharp increases during peak hours rather than gradual changes.

Therefore, tree-based models are more suitable for capturing real-world demand patterns.

## Model Comparison

In [16]:
pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest'],
    'RMSE': [rmse_lr, rmse_rf],
    'R2': [r2_lr, r2_rf]
})

,Model,RMSE,R2
0,Linear Regression,6.684024,0.218582
1,Random Forest,5.668531,0.437985


Random Forest achieves lower RMSE and higher R² compared to Linear Regression.

This indicates that food delivery demand is influenced by non-linear relationships, such as sudden increases during peak hours rather than gradual changes.

Therefore, tree-based models are more suitable for capturing real-world demand patterns.

## Final Insight

The results clearly show that food delivery demand is primarily driven by time-related factors.

Order hour has the strongest impact, reflecting peak consumption periods such as lunch and dinner times.

Weekend effects further highlight differences in customer behavior compared to weekdays.

Weather variables such as temperature and rainfall have a smaller but still noticeable effect on demand.

Overall, this suggests that food delivery demand is more dependent on human behavior than environmental conditions.

From a business perspective, focusing on peak-hour operations and weekend demand planning would be more effective than relying heavily on weather-based adjustments.